In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

project_root = Path.cwd().parent  # repository root (run notebooks from notebooks/)
model_root   = project_root / 'models' / 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart'
orig_root    = project_root / 'models' / 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon'
obs_dir      = project_root / 'data' / 'observations' / 'w'
mapping_file = project_root / 'data' / 'observations' / 'mapping.csv'
tref         = pd.Timestamp('2021-07-12 00:00:00')
hmin         = 0.05   # minimum depth to consider wet [m]

print('model_root exists:', model_root.exists())
print('Files:', [f.name for f in model_root.iterdir() if f.suffix in ('.nc', '.rst', '.log')])

# Warm-start flood simulation — results
**Model:** `ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart`

## Background
The standard SFINCS flood simulations for the Ahr 2021 event start from a completely dry domain (`zsini = 0`).
This is a known limitation for GNN-based surrogates such as mSWE-GNN, which struggle to propagate water
from a dry initial condition during inference.

## Approach
A two-step **warm-start (hotstart)** procedure was applied:

### Step 1 — Spinup run (`spinup_additionalsrc_velocity_100m_cutpolygon`)
A 2-day spinup simulation was run from **10 July 2021 00:00** to **12 July 2021 00:00** using the same
static model setup (grid, bathymetry, roughness) as the velocity model.
Discharge at all 7 source points was held constant at the observed pre-event baseflow values
(taken from `t=0` of the flood simulation's `sfincs.dis`), which already reflect elevated
discharge after approximately 1–2 weeks of antecedent rainfall prior to the flood:

| Source point | Baseflow [m³/s] |
|---|---|
| Kreuzberg | 0.461 |
| Denn | 2.111 |
| Kirmutscheid | 2.780 |
| Muesch | 0.645 |
| new1 / Niederadenau / new2 | 0.000 |
| **Total** | **5.997** |

The spinup was run until steady state and a restart file was written at the end:
`spinup_additionalsrc_velocity_100m_cutpolygon/sfincs.20210712.000000.rst`

### Step 2 — Warm-start flood run (this model)
The flood simulation (12–17 July 2021) was re-run from the spinup restart file by adding:
```
rstfile = ../spinup_additionalsrc_velocity_100m_cutpolygon/sfincs.20210712.000000.rst
```
to `sfincs.inp`. All other settings (grid, discharge forcing, output) are identical to
`ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon`.

## Purpose
This simulation provides physically consistent initial conditions (river channels already carrying
baseflow at `t=0`) for use as training data for the mSWE-GNN surrogate model.
Cell 1 below verifies that wet cells are present at `t=0`.

## 1. Check t=0 water depth (key warm-start check)
If the warm-start worked, water should already be present in the channels at the very first timestep.

In [ ]:
ds_map = xr.open_dataset(model_root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)

time_sec = ds_map['time'].values.astype(float)
map_times = pd.to_datetime(time_sec, unit='s', origin=tref)
ds_map = ds_map.assign_coords(time=map_times)

print('Map time range:', map_times[0], '->', map_times[-1])
print('Number of timesteps:', len(map_times))

zs = ds_map['zs']
zb = ds_map['zb']
da_h = (zs - zb).where(zb > -9999)

# t=0 depth map
h0 = da_h.isel(time=0).values
wet_at_t0 = np.nansum(h0 > hmin)
print(f'\nWet cells at t=0 (depth > {hmin} m): {wet_at_t0}')
print(f'Max depth at t=0: {np.nanmax(h0):.3f} m')

# Build coordinate grid from sfincs.inp parameters
x0_m, y0_m, dx, dy = 344900.0, 5581800.0, 100.0, 100.0
n_rows, n_cols = h0.shape
x_1d = x0_m + (np.arange(n_cols) + 0.5) * dx
y_1d = y0_m + (np.arange(n_rows) + 0.5) * dy
X, Y = np.meshgrid(x_1d, y_1d)

fig, ax = plt.subplots(figsize=(9, 7))
h0_plot = np.where(h0 > hmin, h0, np.nan)
im = ax.pcolormesh(X, Y, h0_plot, cmap='Blues', vmin=0, vmax=3, shading='nearest')
plt.colorbar(im, ax=ax, label='Water depth [m]')
ax.set_title(f'Water depth at t=0 ({map_times[0].strftime("%Y-%m-%d %H:%M")})')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 2. Gauge station time series — warmstart vs original vs observations

In [ ]:
def load_his(root, tref):
    ds = xr.open_dataset(root / 'sfincs_his.nc')
    raw = ds['station_name'].values
    names = []
    for row in raw:
        first = row[0]
        if isinstance(first, (bytes, np.bytes_)):
            names.append(b''.join(row).decode().strip())
        elif isinstance(first, (int, np.integer)):
            names.append(''.join(chr(c) for c in row if c != 0).strip())
        else:
            names.append(''.join(str(c) for c in row).strip())
    times = pd.DatetimeIndex(ds['time'].values)
    zs = ds['point_zs'].values   # (time, stations)
    zb = ds['point_zb'].values   # (stations,)
    depth = zs - zb[np.newaxis, :]
    return times, names, depth

times_ws, names_ws, depth_ws = load_his(model_root, tref)
times_or, names_or, depth_or = load_his(orig_root,  tref)

mapping = pd.read_csv(mapping_file, sep=';')

t_start = pd.Timestamp('2021-07-12 00:00')
t_end   = pd.Timestamp('2021-07-17 00:00')

for i, sname in enumerate(names_ws):
    if sname not in ('Altenahr', 'Bad Bodendorf'):
        continue

    fig, ax = plt.subplots(figsize=(10, 4))

    # Warmstart
    mask_ws = (times_ws >= t_start) & (times_ws <= t_end)
    ax.plot(times_ws[mask_ws], depth_ws[mask_ws, i],
            label='Warmstart', color='steelblue', linewidth=2)

    # Original (dry start)
    if i < depth_or.shape[1]:
        mask_or = (times_or >= t_start) & (times_or <= t_end)
        ax.plot(times_or[mask_or], depth_or[mask_or, i],
                label='Dry start', color='darkorange', linewidth=1.5, linestyle='--')

    # Observations
    row = mapping[mapping['name'] == sname]
    if not row.empty:
        obs_id = str(row['id'].iloc[0])
        obs_files = list(obs_dir.glob(f'{obs_id}_*.csv'))
        if obs_files:
            df_obs = pd.read_csv(obs_files[0], sep=';', decimal=',',
                                 parse_dates=['Datum'], dayfirst=True, encoding='latin1')
            df_obs = df_obs[(df_obs['Datum'] >= t_start) & (df_obs['Datum'] <= t_end)].copy()
            df_obs['Wert'] = df_obs['Wert'] / 100.0
            ax.plot(df_obs['Datum'], df_obs['Wert'], label='Observed', color='black', linewidth=1.5)

    ax.set_title(f'Station {i}: {sname}')
    ax.set_ylabel('Water depth [m]'); ax.set_xlabel('Time')
    ax.grid(True); ax.legend()
    plt.tight_layout(); plt.show()

## 3. Peak flood depth map

In [ ]:
peak_h = da_h.max(dim='time', skipna=True).compute().values
peak_plot = np.where(peak_h > hmin, peak_h, np.nan)

wet_total = int(np.nansum(peak_h > hmin))
print(f'Wet cells (peak > {hmin} m): {wet_total}')
print(f'Peak depth max: {np.nanmax(peak_h):.2f} m')

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.pcolormesh(X, Y, peak_plot, cmap='viridis', vmin=0, vmax=10, shading='nearest')
plt.colorbar(im, ax=ax, label='Peak water depth [m]')
ax.set_title('Peak water depth — warmstart run')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 4. Peak velocity map

In [ ]:
if 'vmax' in ds_map.data_vars:
    vmax_arr = ds_map['vmax'].compute().values
    while vmax_arr.ndim > 2:
        vmax_arr = np.nanmax(vmax_arr, axis=0)
elif 'u' in ds_map.data_vars and 'v' in ds_map.data_vars:
    u = ds_map['u'].compute().values
    v = ds_map['v'].compute().values
    vmax_arr = np.nanmax(np.sqrt(u**2 + v**2), axis=0)
else:
    print('No velocity data found — skipping.')
    vmax_arr = None

if vmax_arr is not None:
    vmax_plot = np.where((peak_h > hmin) & np.isfinite(vmax_arr), vmax_arr, np.nan)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(X, Y, vmax_plot, cmap='plasma', vmin=0, vmax=3, shading='nearest')
    plt.colorbar(im, ax=ax, label='Peak velocity [m/s]')
    ax.set_title('Peak velocity — warmstart run')
    ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
    ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

## 5. Discharge through cross-sections downstream of source points

For each gauged source point, a cross-section is placed **300 m downstream** in the direction of the gauge station.
Discharge is integrated across the section as:

$$Q(t) = \sum_i h_i(t) \cdot u_{n,i}(t) \cdot \Delta l$$

where $u_n = u \cdot f_x + v \cdot f_y$ is the velocity component normal to the section (in the flow direction) and $\Delta l = 100$ m is the cell width.

In [ ]:
# =============================================================================
# Discharge through QGIS cross-sections vs observed / injected Q
#
# Wet-width correction:
#   wet_frac = Q_observed_gauge(t=0) / Q_computed_section(t=0)
#
# Using the gauge observation at t=0 (not the single-tributary injection) as
# the reference, because the sections sit on the main Ahr and carry the total
# cumulative discharge from all upstream source injections.
# For ungauged stations (new1, Niederadenau, new2) the median factor from the
# four gauged stations is applied as a proxy.
# =============================================================================
import os
import geopandas as gpd

os.environ['SHAPE_RESTORE_SHX'] = 'YES'

shp_path  = project_root / 'data' / 'observations' / 'sections_discharge_computation.shp'
obs_dir_q = project_root / 'data' / 'observations' / 'q'
src_csv   = project_root / 'data' / 'observations' / 'src_points_ts_Marg_additionalsrc.csv'

cell_size = 100.0
t0        = pd.Timestamp('2021-07-12 00:00')
t_start_q = pd.Timestamp('2021-07-12 00:00')
t_end_q   = pd.Timestamp('2021-07-17 00:00')

src_info = {
    'Kreuzberg':    {'xy': (355971.4,  5597633.0),  'obs_id': '2718090200', 'csv_col': None},
    'Denn':         {'xy': (356927.2,  5594304.1),  'obs_id': '2718070900', 'csv_col': None},
    'Kirmutscheid': {'xy': (346313.8,  5582100.3),  'obs_id': '2718050500', 'csv_col': None},
    'Müsch':        {'xy': (345220.8,  5583766.9),  'obs_id': '2718010800', 'csv_col': None},
    'new1':         {'xy': (348791.19, 5590885.31), 'obs_id': None,         'csv_col': '10'},
    'Niederadenau': {'xy': (353114.5,  5589087.4),  'obs_id': None,         'csv_col': '11'},
    'new2':         {'xy': (352987.3,  5592047.304),'obs_id': None,         'csv_col': '12'},
}

sections = gpd.read_file(str(shp_path))
if sections.crs is None:
    sections = sections.set_crs('EPSG:32632')

section_geom = {}
for _, row in sections.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    name = min(src_info, key=lambda n: np.hypot(cx - src_info[n]['xy'][0],
                                                 cy - src_info[n]['xy'][1]))
    section_geom[name] = row.geometry

df_src = pd.read_csv(str(src_csv), parse_dates=['Datum']).set_index('Datum')
df_src = df_src[(df_src.index >= t_start_q) & (df_src.index <= t_end_q)]

print('Loading u, v, h arrays...')
u_arr = ds_map['u'].values
v_arr = ds_map['v'].values
h_arr = da_h.values
msk   = ds_map['msk'].values
if msk.ndim == 3:
    msk = msk[0]
print('Done.\n')


def integrate_section_Q(geom, sx, sy):
    coords   = np.array(geom.coords)
    start, end = coords[0], coords[-1]
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)
    lx, ly   = line_vec / line_len
    cx, cy   = geom.centroid.x, geom.centroid.y
    for nx, ny in [(-ly, lx), (ly, -lx)]:
        if (cx - sx) * nx + (cy - sy) * ny > 0:
            break
    n_samples = max(2, int(line_len / cell_size) + 1)
    seen, cell_idxs = set(), []
    for d in np.linspace(0, line_len, n_samples):
        pt = start + d * np.array([lx, ly])
        dist2 = (X - pt[0]) ** 2 + (Y - pt[1]) ** 2
        ni, mi = np.unravel_index(np.argmin(dist2), dist2.shape)
        if (ni, mi) not in seen and msk[ni, mi] != 0:
            seen.add((ni, mi))
            cell_idxs.append((ni, mi))
    Q_t = np.zeros(len(map_times))
    for ni, mi in cell_idxs:
        h_t = np.where(np.isfinite(h_arr[:, ni, mi]), h_arr[:, ni, mi], 0.0)
        u_t = np.where(np.isfinite(u_arr[:, ni, mi]), u_arr[:, ni, mi], 0.0)
        v_t = np.where(np.isfinite(v_arr[:, ni, mi]), v_arr[:, ni, mi], 0.0)
        Q_t += h_t * (u_t * nx + v_t * ny) * cell_size
    return Q_t, len(cell_idxs)


# ── Step 1: compute correction factors from gauged stations ──────────────────
print(f"{'Station':<15}  {'Q_obs(t=0)':>10}  {'Q_comp(t=0)':>12}  {'wet_frac':>9}")
print('-' * 55)

wet_fracs_gauged = {}
section_Q_raw = {}   # store raw Q arrays for reuse

for name, geom in section_geom.items():
    info   = src_info[name]
    sx, sy = info['xy']
    Q_t, n_cells = integrate_section_Q(geom, sx, sy)
    section_Q_raw[name] = Q_t

    if info['obs_id'] is not None:
        q_files = list(obs_dir_q.glob(f"{info['obs_id']}_*.csv"))
        if q_files:
            df_q = pd.read_csv(q_files[0], sep=';', decimal=',',
                               parse_dates=['Datum'], dayfirst=True, encoding='latin1')
            # Q_obs at t0: nearest available timestamp
            df_q = df_q.set_index('Datum').sort_index()
            df_q = df_q[~df_q.index.duplicated(keep='first')]
            idx_t0 = df_q.index.get_indexer([t0], method='nearest')[0]
            Q_obs_t0 = float(df_q['Wert'].iloc[idx_t0])
            Q_comp_t0 = Q_t[0]
            frac = Q_obs_t0 / Q_comp_t0 if Q_comp_t0 > 0 else 1.0
            wet_fracs_gauged[name] = frac
            print(f"{name:<15}  {Q_obs_t0:>10.3f}  {Q_comp_t0:>12.3f}  {frac:>9.3f}")

median_frac = float(np.median(list(wet_fracs_gauged.values())))
print(f"\nMedian wet fraction (gauged): {median_frac:.3f}  → used for ungauged stations\n")

# ── Step 2: plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(section_geom), 1,
                         figsize=(11, 3 * len(section_geom)), sharex=True)
if len(section_geom) == 1:
    axes = [axes]

for ax, (name, geom) in zip(axes, section_geom.items()):
    info = src_info[name]

    Q_t    = section_Q_raw[name]
    frac   = wet_fracs_gauged.get(name, median_frac)
    Q_corr = Q_t * frac

    Q_s = pd.Series(Q_corr, index=map_times)
    Q_s = Q_s[(Q_s.index >= t_start_q) & (Q_s.index <= t_end_q)]

    label_src = 'obs-calibrated' if name in wet_fracs_gauged else f'median f={median_frac:.2f}'
    ax.plot(Q_s.index, Q_s.values, color='steelblue', linewidth=2,
            label=f'Simulated Q ({label_src}, f={frac:.2f})')

    if info['obs_id'] is not None:
        q_files = list(obs_dir_q.glob(f"{info['obs_id']}_*.csv"))
        if q_files:
            df_q = pd.read_csv(q_files[0], sep=';', decimal=',',
                               parse_dates=['Datum'], dayfirst=True, encoding='latin1')
            df_q = df_q[(df_q['Datum'] >= t_start_q) & (df_q['Datum'] <= t_end_q)]
            ax.plot(df_q['Datum'], df_q['Wert'], color='black', linewidth=1.5,
                    label='Observed Q [m³/s]')
    elif info['csv_col'] is not None:
        ax.plot(df_src.index, df_src[info['csv_col']], color='black',
                linewidth=1.5, linestyle='--', label='Injected Q (sfincs.dis)')

    ax.axhline(0, color='k', linewidth=0.5, linestyle=':')
    ax.set_title(name)
    ax.set_ylabel('Q [m³/s]')
    ax.legend(fontsize=8)
    ax.grid(True)

axes[-1].set_xlabel('Time')
plt.suptitle('Simulated vs reference discharge — gauge-calibrated wet-width correction\n'
             '(warmstart, 100 m grid)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Discharge overestimation diagnostic
# At t=0 (warm-start steady state) the true Q at each section ≈ injected baseflow.
# The ratio Q_computed(t=0) / Q_injected(t=0) quantifies the overestimation factor
# caused by using the full 100 m cell width instead of the actual wet channel width.
# =============================================================================

# Injected baseflow at t=0 from sfincs.dis (src_points_ts_Marg_additionalsrc.csv, row 0)
baseflow_q = {
    'Kreuzberg':    0.461,
    'Denn':         2.111,
    'Kirmutscheid': 2.780,
    'Müsch':        0.645,
    # new1 / Niederadenau / new2 had 0.1 m³/s symbolic flow during spinup but 0.0 in flood sim
}

print(f"{'Station':<15}  {'Q_computed t=0':>15}  {'Q_injected t=0':>15}  {'factor':>8}  {'est. wet width (m)':>20}")
print("-" * 80)

for name, geom in section_geom.items():
    if name not in baseflow_q:
        continue
    info   = src_info[name]
    sx, sy = info['xy']
    Q_t, _ = integrate_section_Q(geom, sx, sy)
    Q_t0   = Q_t[0]
    Q_inj  = baseflow_q[name]
    factor = Q_t0 / Q_inj if Q_inj > 0 else float('nan')
    # Estimated actual wet width = cell_size / factor  (per active cell)
    wet_w = ('~' + str(round(cell_size / factor))) if (factor > 0 and factor != float('inf')) else 'n/a'
    print(f"{name:<15}  {Q_t0:>15.3f}  {Q_inj:>15.3f}  {factor:>8.1f}  {wet_w:>20}")

In [ ]:
# =============================================================================
# Map of QGIS cross-section positions
# Requires: peak_h, X, Y, section_geom, src_info from discharge cell above
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 8))

peak_plot = np.where(peak_h > hmin, peak_h, np.nan)
im = ax.pcolormesh(X, Y, peak_plot, cmap='Blues', vmin=0, vmax=8,
                   shading='nearest', alpha=0.7)
plt.colorbar(im, ax=ax, label='Peak water depth [m]', shrink=0.6)

for name, geom in section_geom.items():
    coords = np.array(geom.coords)
    sx, sy = src_info[name]['xy']

    # Draw section line
    ax.plot(coords[:, 0], coords[:, 1], color='red', linewidth=2.5, zorder=4)
    ax.plot(geom.centroid.x, geom.centroid.y, 'r+', markersize=8, zorder=5)

    # Source point
    ax.plot(sx, sy, 'g^', markersize=9, zorder=5)

    ax.annotate(name, xy=(geom.centroid.x, geom.centroid.y),
                xytext=(5, 4), textcoords='offset points',
                fontsize=8, color='darkred', zorder=6)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='red', linewidth=2.5, label='Cross-section (QGIS)'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='green',
           markersize=9, label='Source point'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)
ax.set_xlabel('Easting [m]')
ax.set_ylabel('Northing [m]')
ax.set_title('QGIS cross-section positions over peak flood depth\n(warmstart, 100 m grid)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Compat shim: this hydromt version's io.py calls pd.read_csv(..., delim_whitespace=True),
# an argument removed from the installed pandas. Translate it to sep=r"\s+" instead.
_orig_read_csv = pd.read_csv


def _read_csv_compat(*args, **kwargs):
    if kwargs.pop("delim_whitespace", False):
        kwargs.setdefault("sep", r"\s+")
    return _orig_read_csv(*args, **kwargs)


pd.read_csv = _read_csv_compat

try:
    from hydromt_sfincs import SfincsModel
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "hydromt-sfincs"])
    from hydromt_sfincs import SfincsModel

# plot_basemap(bmap="sat") needs cartopy for the satellite tile background
try:
    import cartopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "cartopy"])

# Load the model for satellite-basemap plotting (sf_sim is only used for plot_basemap)
sf_sim = SfincsModel(root=str(model_root), mode="r")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from pathlib import Path

# Keep embedded animation manageable in notebooks
max_frames = 120
step = max(1, int(np.ceil(da_h.time.size / max_frames)))
cbar_kwargs = {"shrink": 0.6, "anchor": (0, 0)}
plt.rcParams["animation.embed_limit"] = 200  # MB

# Fixed color scale for the whole animation, sized to the run's true peak depth
# (peak_h from Section 3) so it never rescales frame-to-frame.
vmin_h, vmax_h = 0, float(np.nanmax(peak_h))


def update_plot(i, da_h, cax_h):
    da_hi = da_h.isel(time=i)
    t = da_hi.time.dt.strftime("%d-%B-%Y %H:%M:%S").item()
    ax.set_title(f"SFINCS water depth {t}")
    cax_h.set_array(da_hi.values.ravel())
    cax_h.set_clim(vmin_h, vmax_h)  # re-assert fixed scale every frame


fig, ax = sf_sim.plot_basemap(
    fn_out=None,
    variable="",
    bmap="sat",
    plot_bounds=False,
    figsize=(11, 7),
    zoomlevel=12,
    )
cax_h = da_h.isel(time=0).plot(
    x="x", y="y", ax=ax, vmin=vmin_h, vmax=vmax_h, cmap=plt.cm.viridis, cbar_kwargs=cbar_kwargs
    )
plt.close()  # prevent duplicate static plot

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames=np.arange(0, da_h.time.size, step),
    interval=250,  # ms between frames
    fargs=(
        da_h,
        cax_h,
    ),
    )

# Save animation with writer-aware extension.
figs_dir = Path(model_root) / "figs"
figs_dir.mkdir(parents=True, exist_ok=True)
ffmpeg_ok = animation.writers.is_available("ffmpeg")
writer = "ffmpeg" if ffmpeg_ok else "pillow"
ext = "mp4" if ffmpeg_ok else "gif"
save_path = figs_dir / f"sfincs_h.{ext}"
try:
    ani.save(str(save_path), writer=writer, fps=4, dpi=200)
    print(f"Saved animation to: {save_path}")
except Exception as e:
    print(f"Could not save animation to {save_path}: {e}")

# Show in notebook using JS-based renderer
from IPython.display import HTML

HTML(ani.to_jshtml())

## 6. Average residence time of water in the catchment

Estimated with the reservoir-theory (bulk) definition of mean residence/transit time:

$$\tau(t) = \frac{V_{storage}(t)}{Q_{out}(t)}$$

- $V_{storage}(t)$: total volume of water held in the domain at each output timestep, $\sum h \cdot \Delta x \Delta y$ over wet cells (from the model — there's no way to observe this directly).
- $Q_{out}(t)$: discharge leaving the catchment, taken directly from the **observed** Bad Bodendorf gauge record (id `2718060700`, 15-min resolution, interpolated to the model's hourly output times), rather than computed from the model's own velocity field.

Why observed instead of modeled discharge: a cross-section integrated from the model (h·u·Δl over a 100 m grid cell) overestimates true channel discharge, because the real channel is narrower than one grid cell — the same bias the notebook's own Section 5 already identified and corrected for with a gauge-calibrated scale factor. That correction is only calibrated at t=0 baseflow, though, and turns out to break down badly at high flow: the corrected model cross-section peaks at only ~294 m³/s, while the observed gauge peaks at ~1050 m³/s. Using the observed record directly sidesteps that bias entirely for the part of τ that we can measure directly, and only leans on the model for the part we can't (spatially distributed storage). The raw (uncorrected) model cross-section is still plotted below for comparison, but the analysis itself uses the observed series as $Q_{out}(t)$.

This is still an approximation (assumes the domain behaves like a well-mixed store), not a true Lagrangian particle average — and it only covers the simulated window (12–17 July); if storage hasn't returned to its t=0 level by the end, the event-average below is a lower bound on the true residence time.

In [ ]:
cell_size = 100.0
cell_area = cell_size ** 2  # 100 m x 100 m grid cells [m^2]

# --- Storage volume V(t): total water volume held in the domain (model-only,
#     nothing to observe this against) ---
h_wet = xr.where(da_h > hmin, da_h, 0.0)
V_storage = (h_wet * cell_area).sum(dim=('n', 'm')).values  # [m^3]

# --- Modeled outlet cross-section: kept only as a diagnostic comparison,
#     NOT used to drive tau (see markdown above for why) ---
msk_arr = ds_map['msk'].values
if msk_arr.ndim == 3:
    msk_arr = msk_arr[0]
active_cols = np.where(msk_arr.sum(axis=0) > 0)[0]
outlet_col = active_cols.max() - 10  # ~1 km inside the eastern domain edge
outlet_rows = np.where(msk_arr[:, outlet_col] > 0)[0]

u_arr = ds_map['u'].values  # (time, n, m)
h_arr = da_h.values

Q_out_model = np.zeros(len(map_times))
for r in outlet_rows:
    h_t = np.where(np.isfinite(h_arr[:, r, outlet_col]) & (h_arr[:, r, outlet_col] > hmin),
                    h_arr[:, r, outlet_col], 0.0)
    u_t = np.where(np.isfinite(u_arr[:, r, outlet_col]), u_arr[:, r, outlet_col], 0.0)
    Q_out_model += h_t * u_t * cell_size  # [m^3/s]

# --- Observed outlet discharge (Bad Bodendorf gauge, id 2718060700): this is
#     the Q_out(t) actually used for tau. 15-min observations interpolated
#     onto the model's hourly output timestamps. ---
obs_dir_q = project_root / 'data' / 'observations' / 'q'
q_files = list(obs_dir_q.glob('2718060700_*.csv'))
df_q_out = pd.read_csv(q_files[0], sep=';', decimal=',',
                       parse_dates=['Datum'], dayfirst=True, encoding='latin1')
df_q_out = df_q_out.set_index('Datum').sort_index()
df_q_out = df_q_out[~df_q_out.index.duplicated(keep='first')]

# force both time axes to the same (ns) integer resolution before interpolating
xp = df_q_out.index.values.astype('datetime64[ns]').astype(np.int64)
x = pd.DatetimeIndex(map_times).values.astype('datetime64[ns]').astype(np.int64)
Q_out = np.interp(x, xp, df_q_out['Wert'].values)  # [m^3/s]

print(f'Q_out(t=0): observed = {Q_out[0]:.2f} m^3/s, '
      f'model cross-section (uncorrected) = {Q_out_model[0]:.2f} m^3/s')
print(f'Peak Q_out: observed = {Q_out.max():.0f} m^3/s, '
      f'model cross-section (uncorrected) = {Q_out_model.max():.0f} m^3/s')

# --- Residence time tau(t) = V_storage(t) / Q_out(t) ---
Q_min_valid = 0.5  # m3/s; below this, tau is numerically meaningless/noisy
valid = Q_out > Q_min_valid
tau_hours = np.full(len(map_times), np.nan)
tau_hours[valid] = V_storage[valid] / Q_out[valid] / 3600.0

# --- Summary ---
mask_event = (map_times >= t_start) & (map_times <= t_end)
times_event = map_times[mask_event]
tau_event = tau_hours[mask_event]
Qout_event = Q_out[mask_event]
tau_event_valid = tau_event[np.isfinite(tau_event)]

print(f'\nStorage volume at t=0:   {V_storage[0]:,.0f} m^3')
print(f'Storage volume at t=end: {V_storage[-1]:,.0f} m^3  '
      f'({V_storage[-1] / V_storage[0]:.1f}x the t=0 value)')
if V_storage[-1] > 1.2 * V_storage[0]:
    print('WARNING: storage at end of run is well above t=0 level — the domain has '
          'NOT fully drained within the simulated 5-day window. The event-average '
          'below is therefore a LOWER BOUND on the true residence time, not the '
          'full drainage/recession time.')

print(f'\nAverage residence time over event window: {np.nanmean(tau_event_valid):.2f} h '
      f'({np.nanmean(tau_event_valid) / 24:.2f} d)')
print(f'Median residence time over event window:  {np.nanmedian(tau_event_valid):.2f} h '
      f'({np.nanmedian(tau_event_valid) / 24:.2f} d)  (more robust to peak-time noise)')

# --- Shortest / longest tau in the series ---
# NOTE: tau(t) is one lumped domain-average number per timestep, not a
# distribution of individual water-parcel travel times, so "shortest"/"longest"
# below mean the extremes of that time-varying average, not a fastest/slowest
# parcel. Genuine per-parcel min/max transit times would need Lagrangian
# particle tracking through the u/v velocity field.
idx_valid = np.where(np.isfinite(tau_event))[0]
i_min = idx_valid[np.argmin(tau_event[idx_valid])]
i_max = idx_valid[np.argmax(tau_event[idx_valid])]

print(f'\nShortest τ in series: {tau_event[i_min]:.1f} h ({tau_event[i_min] / 24:.2f} d) '
      f'at {times_event[i_min]} (Q_out={Qout_event[i_min]:.0f} m^3/s).')
print('  -> Falls right at peak observed discharge: the domain flushing fastest under '
      'peak flood-flow conditions. The one genuinely meaningful extreme here.')

print(f'\nLongest τ in series:  {tau_event[i_max]:.1f} h ({tau_event[i_max] / 24:.2f} d) '
      f'at {times_event[i_max]} (Q_out={Qout_event[i_max]:.1f} m^3/s).')
print('  -> Falls during quiet low-flow conditions before the flood pulse arrives. This '
      'is NOT a real signal: at low flow, the observed gauge discharge is quantized by '
      'its rating-curve lookup table (visible as repeated/stair-stepped values), while '
      'storage varies smoothly — dividing a smooth number by a noisy/quantized one '
      'amplifies that quantization into artificial wiggles in tau, and this point is '
      'essentially where the noise floor happened to peak. Combined with the drainage '
      'warning above, the domain has not finished draining by the end of this run either, '
      'so the TRUE longest residence time is still unknown — it requires either extending '
      'the simulation until storage returns to its t=0 level, or Lagrangian particle '
      'tracking to find genuinely slow (e.g. stagnant floodplain) parcels.')

# --- Low-flow band: before the flood pulse arrives, Q_out is dominated by
#     gauge rating-curve quantization noise (see note above) rather than a
#     real physical signal, so tau there is unreliable point-by-point.
low_flow_thresh = 2.5 * Q_out[0]  # ~baseflow noise band
above_thresh = Qout_event > low_flow_thresh
low_flow_end_idx = int(np.argmax(above_thresh)) if np.any(above_thresh) else len(above_thresh) - 1
t_low_flow_end = times_event[low_flow_end_idx]
print(f'\nLow-flow / noise-dominated period: {times_event[0]} to {t_low_flow_end} '
      f'(Q_out < {low_flow_thresh:.1f} m^3/s) — shaded gray below.')

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(map_times, V_storage / 1e6, color='steelblue')
axes[0].set_ylabel('Storage volume\n[10^6 m^3]')
axes[0].grid(True)

axes[1].plot(map_times, Q_out_model, color='lightgray', linestyle='--',
             label='model cross-section (uncorrected, for comparison)')
axes[1].plot(map_times, Q_out, color='darkorange', label='observed (Bad Bodendorf, used for τ)')
axes[1].set_ylabel('Outlet discharge\nQ_out [m^3/s]')
axes[1].legend(fontsize=8)
axes[1].grid(True)

axes[2].plot(map_times, tau_hours, color='seagreen')
axes[2].scatter([times_event[i_min]], [tau_event[i_min]], color='blue', zorder=5,
                label='shortest (peak flow — meaningful)')
axes[2].scatter([times_event[i_max]], [tau_event[i_max]], color='red', marker='x', zorder=5,
                label='longest in series (noise artifact, see note)')
axes[2].set_ylabel('Residence time\nτ [hours]')
axes[2].set_xlabel('Time')
axes[2].legend(fontsize=8)
axes[2].grid(True)

for ax in axes:
    ax.set_xlim(t_start, t_end)
    ax.axvspan(times_event[0], t_low_flow_end, color='gray', alpha=0.15, zorder=0)
axes[2].text(times_event[0], axes[2].get_ylim()[1] * 0.97,
             '  low-flow: gauge-quantization noise,\n  don\'t read point values here',
             fontsize=7, va='top', color='dimgray')

plt.suptitle('Residence time of water in the catchment (τ = V_storage / observed Q_out)')
plt.tight_layout()
plt.show()

## 7. Peak water depth error and peak timing offset (Altenahr, Bad Bodendorf)

For each station, the warmstart simulation (Section 2) is compared against the observed water depth record over the event window (12–17 July 2021):
- **Peak water depth error**: difference between the simulated and observed peak depth (each found independently on its own series), in m and as a percentage of the observed peak.
- **Peak timing offset**: time difference between the simulated and observed peaks, in hours (positive = simulated peak arrives later than observed).
- **RMSE / NSE**: computed over the full window on paired timestamps, by linearly interpolating the observed record onto the simulation's hourly output times.</cell_type>
</invoke>


In [ ]:
#peak water depth error and peak timing off set
# =============================================================================
# Peak water depth error and peak timing offset — Altenahr & Bad Bodendorf
# Reuses times_ws / names_ws / depth_ws (warmstart) and mapping / obs_dir
# from Section 2.
# =============================================================================

results = {}

for sname in ('Altenahr', 'Bad Bodendorf'):
    i = names_ws.index(sname)

    mask_ws = (times_ws >= t_start) & (times_ws <= t_end)
    t_sim = times_ws[mask_ws]
    h_sim = depth_ws[mask_ws, i]

    row = mapping[mapping['name'] == sname]
    obs_id = str(row['id'].iloc[0])
    obs_files = list(obs_dir.glob(f'{obs_id}_*.csv'))
    df_obs = pd.read_csv(obs_files[0], sep=';', decimal=',',
                         parse_dates=['Datum'], dayfirst=True, encoding='latin1')
    df_obs = df_obs[(df_obs['Datum'] >= t_start) & (df_obs['Datum'] <= t_end)].copy()
    df_obs['Wert'] = df_obs['Wert'] / 100.0
    df_obs = df_obs.sort_values('Datum')
    t_obs = df_obs['Datum'].values
    h_obs = df_obs['Wert'].values

    # Interpolate observed onto simulated (hourly) timestamps for paired RMSE/NSE
    xp = t_obs.astype('datetime64[ns]').astype(np.int64)
    x  = t_sim.values.astype('datetime64[ns]').astype(np.int64)
    h_obs_interp = np.interp(x, xp, h_obs)

    rmse = float(np.sqrt(np.mean((h_sim - h_obs_interp) ** 2)))
    nse  = float(1 - np.sum((h_obs_interp - h_sim) ** 2)
                     / np.sum((h_obs_interp - h_obs_interp.mean()) ** 2))

    # Peak magnitude/timing, each found independently on its own (native-resolution) series
    i_peak_sim = int(np.argmax(h_sim))
    i_peak_obs = int(np.argmax(h_obs))
    peak_sim, t_peak_sim = float(h_sim[i_peak_sim]), pd.Timestamp(t_sim[i_peak_sim])
    peak_obs, t_peak_obs = float(h_obs[i_peak_obs]), pd.Timestamp(t_obs[i_peak_obs])

    peak_err_m    = peak_sim - peak_obs
    peak_err_pct  = peak_err_m / peak_obs * 100.0
    peak_offset_h = (t_peak_sim - t_peak_obs).total_seconds() / 3600.0

    results[sname] = dict(peak_sim=peak_sim, t_peak_sim=t_peak_sim,
                           peak_obs=peak_obs, t_peak_obs=t_peak_obs,
                           peak_err_m=peak_err_m, peak_err_pct=peak_err_pct,
                           peak_offset_h=peak_offset_h, rmse=rmse, nse=nse)

    print(f'--- {sname} ---')
    print(f'  Peak simulated: {peak_sim:.3f} m at {t_peak_sim}')
    print(f'  Peak observed:  {peak_obs:.3f} m at {t_peak_obs}')
    print(f'  Peak WD error:  {peak_err_m:+.3f} m ({peak_err_pct:+.1f} %)')
    print(f'  Peak timing offset: {peak_offset_h:+.1f} h (positive = simulated peak is later)')
    print(f'  RMSE (full window): {rmse:.3f} m')
    print(f'  NSE  (full window):  {nse:.3f}')
    print()

pd.DataFrame(results).T[['peak_sim', 'peak_obs', 'peak_err_m', 'peak_err_pct',
                          'peak_offset_h', 'rmse', 'nse']]